# exp002: CV安定性確認（seed違いマルチシード検証）

数値7カラム・LightGBM・StratifiedKFold 5-fold を5つのseedで実行し、seed間のval scoreばらつきを計測する。

ローカル実行結果（OOF mean±std = 0.44477 ± 0.00053）との比較用。

In [ ]:
"""
exp002: CV安定性確認（seed違いマルチシード検証）— Kaggle Notebook 自己完結版

ローカル版 experiments/runs/exp002_s1_cv_stability_multiseed.py と同じロジックを、
Kaggle Dataset 経由の import に依存せず単一ファイルに埋め込んだもの。
Kaggle Notebook 上でコンペデータ（/kaggle/input/<competition>/）を直接読み込んで実行する。

ローカルで得た結果（seed間std=0.00053, OOF mean=0.44477）とKaggleGPU実行結果を突き合わせ、
環境間で再現するかを確認する。
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from pathlib import Path

# ── コンペデータのパス自動検出 ──────────────────────
COMPETITION = "playground-series-s6e7"
_KAGGLE_INPUT = Path("/kaggle/input")
_comp_candidates = [
    _KAGGLE_INPUT / "competitions" / COMPETITION,
    _KAGGLE_INPUT / COMPETITION,
]
RAW_DATA_DIR = next((p for p in _comp_candidates if p.exists()), _comp_candidates[0])
print(f"RAW_DATA_DIR = {RAW_DATA_DIR}")

TARGET_COL = "health_condition"
N_CLASSES = 3
N_SPLITS = 5
SEEDS = [42, 43, 44, 45, 46]

FEATURES = [
    "sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
    "step_count", "exercise_duration", "water_intake",
]

LGB_PARAMS = {
    "objective": "multiclass",
    "num_class": N_CLASSES,
    "metric": "multi_logloss",
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "num_leaves": 63,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "verbose": -1,
}


def preprocess():
    train = pd.read_csv(RAW_DATA_DIR / "train.csv")
    test = pd.read_csv(RAW_DATA_DIR / "test.csv")
    medians = train[FEATURES].median()
    train[FEATURES] = train[FEATURES].fillna(medians)
    test[FEATURES] = test[FEATURES].fillna(medians)
    return train, test


def train_fold_lgb(X_tr, y_tr, X_val, y_val, params):
    import lightgbm as lgb
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)],
    )
    return model, model.predict_proba(X_val)


def run_cv(train, test, seed):
    X, y_raw = train[FEATURES], train[TARGET_COL]
    X_test = test[FEATURES]

    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y_raw), index=y_raw.index)

    params = {**LGB_PARAMS, "random_state": seed}
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    oof_preds = np.zeros((len(train), N_CLASSES))
    train_scores, val_scores = [], []

    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model, val_pred = train_fold_lgb(X_tr, y_tr, X_val, y_val, params)
        oof_preds[val_idx] = val_pred

        tr_score = balanced_accuracy_score(y_tr, model.predict(X_tr))
        val_score = balanced_accuracy_score(y_val, np.argmax(val_pred, axis=1))
        train_scores.append(tr_score)
        val_scores.append(val_score)

    oof_score = balanced_accuracy_score(y, np.argmax(oof_preds, axis=1))
    return {"train_scores": train_scores, "val_scores": val_scores, "oof_score": oof_score}


def main():
    train, test = preprocess()
    print(f"train: {train.shape}, test: {test.shape}")

    oof_scores = []
    all_train_scores, all_val_scores = [], []

    for seed in SEEDS:
        result = run_cv(train, test, seed=seed)
        train_mean = float(np.mean(result["train_scores"]))
        val_mean = float(np.mean(result["val_scores"]))
        all_train_scores.extend(result["train_scores"])
        all_val_scores.extend(result["val_scores"])
        oof_scores.append(result["oof_score"])
        print(f"seed={seed}: train_mean={train_mean:.5f}  val_mean={val_mean:.5f}  "
              f"oof={result['oof_score']:.5f}  folds={[f'{s:.5f}' for s in result['val_scores']]}")

    val_std_across_seeds = float(np.std(oof_scores))
    val_std_across_folds = float(np.std(all_val_scores))

    print(f"\n=== seed間ばらつき（Kaggle Notebook実行）===")
    print(f"OOF scores (5 seeds): {[f'{s:.5f}' for s in oof_scores]}")
    print(f"OOF mean±std across seeds: {np.mean(oof_scores):.5f} ± {val_std_across_seeds:.5f}")
    print(f"全fold(25個)の val std: {val_std_across_folds:.5f}")
    print(f"\n=== ローカル実行結果との比較用 ===")
    print(f"ローカル: OOF mean±std = 0.44477 ± 0.00053, train_mean=0.46555±0.00515, val_mean=0.44477±0.00188")
    print(f"Kaggle : OOF mean±std = {np.mean(oof_scores):.5f} ± {val_std_across_seeds:.5f}, "
          f"train_mean={np.mean(all_train_scores):.5f}±{np.std(all_train_scores):.5f}, "
          f"val_mean={np.mean(all_val_scores):.5f}±{np.std(all_val_scores):.5f}")


if __name__ == "__main__":
    main()
